# Debug — Calcul risque journalier
Visualisation pas-à-pas de la répartition IFT sur les périodes de traitement et du calcul de risque.

In [ ]:
import sys
sys.path.insert(0, '../../etl')

import polars as pl
import pandas as pd
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import date, timedelta
from calcul_risque_journalier import CULTURE_MAPPING, normaliser_culture

PARQUET = '../../data/parquet'
ANNEE   = 2025
INSEE   = '37031'   # Bourgueil — changer ici pour tester une autre commune

## 1. Données IFT de la commune

In [ ]:
ift_raw = pl.read_parquet(f'{PARQUET}/ift_communes_enrichi.parquet')
commune = ift_raw.filter(pl.col('insee_com') == INSEE)

old, new = list(CULTURE_MAPPING.keys()), list(CULTURE_MAPPING.values())
commune = commune.with_columns([
    pl.col('c_maj').replace_strict(old=old, new=new, default=None).alias('c_maj_cal'),
    pl.col('c_ift_hbc').replace_strict(old=old, new=new, default=None).alias('c_ift_hbc_cal'),
    pl.col('c_ift_h').replace_strict(old=old, new=new, default=None).alias('c_ift_h_cal'),
])

r = commune.to_pandas().iloc[0]
dep = r['code_insee_dep']
print(f"Commune  : {r.get('nom_commune', INSEE)} ({INSEE}) — dép. {dep}")
print()
print(f"{'':40s} {'Culture IFT':30s}  {'→ Calendrier':25s}  {'IFT annuel':>10}")
print('-' * 110)
for col_raw, col_cal, ift_col, type_ttt in [
    ('c_maj',     'c_maj_cal',     'ift_t_hbc',  'Total (toutes périodes)'),
    ('c_ift_hbc', 'c_ift_hbc_cal', 'ift_hh_hbc', 'Fongi/insecti uniquement'),
    ('c_ift_h',   'c_ift_h_cal',   'ift_h',      'Herbicides uniquement'),
]:
    raw = r.get(col_raw, '—')
    cal = r.get(col_cal)
    ift_val = r.get(ift_col, 0) or 0
    cal_str = str(cal) if pd.notna(cal) else '⚠ HORS CALENDRIER'
    print(f"  {type_ttt:38s} {str(raw):30s}  {cal_str:25s}  {ift_val:>10.3f}")

## 2. Calendrier d'épandage pour cette commune (normalisé vers l'année cible)

In [ ]:
con = duckdb.connect(':memory:')
cal_df = con.execute(f"""
    SELECT
        culture,
        make_date({ANNEE}, month(Debut_de_periode), day(Debut_de_periode)) AS debut,
        make_date({ANNEE}, month(Fin_de_periode),   day(Fin_de_periode))   AS fin,
        Herbicides, Fongicides, Insecticides,
        COUNT(*) FILTER (WHERE Herbicides)
            OVER (PARTITION BY culture) AS nb_herb,
        COUNT(*) FILTER (WHERE Fongicides OR Insecticides)
            OVER (PARTITION BY culture) AS nb_fongi_insecti,
        COUNT(*) OVER (PARTITION BY culture) AS nb_total
    FROM read_parquet('{PARQUET}/calendrier_epandage.parquet')
    WHERE departement_code = {int(dep)}
""").df()
con.close()

# Filtrer sur les cultures de la commune
cultures_commune = {r.get(c) for c in ('c_maj_cal','c_ift_hbc_cal','c_ift_h_cal') if pd.notna(r.get(c))}
print(f"Cultures recherchées dans le calendrier : {cultures_commune}")
print()
cal_commune = cal_df[cal_df['culture'].isin(cultures_commune)].sort_values(['culture','debut'])
display(cal_commune)

## 3. IFT journalier — répartition sur les périodes

In [ ]:
# Pour chaque couple (culture, type_traitement), calculer l'IFT par période
configs = [
    ('c_maj_cal',     'ift_t_hbc',  None,                           'nb_total',         'Total (c_maj)'),
    ('c_ift_hbc_cal', 'ift_hh_hbc', lambda df: df['Fongicides'] | df['Insecticides'], 'nb_fongi_insecti', 'Fongi/Insecti (c_ift_hbc)'),
    ('c_ift_h_cal',   'ift_h',      lambda df: df['Herbicides'],   'nb_herb',          'Herbicides (c_ift_h)'),
]

print(f"{'Config':28s} {'Culture cal':20s} {'IFT annuel':>10} {'Nb périodes':>12} {'IFT/période':>12}")
print('-' * 88)

for col_cal, ift_col, filtre, col_nb, label in configs:
    culture = r.get(col_cal)
    ift_val = r.get(ift_col, 0) or 0
    
    if pd.isna(culture):
        print(f"  {label:26s} {'⚠ hors calendrier':20s} {ift_val:>10.3f} {'—':>12} {'—':>12}")
        continue
    
    periodes = cal_commune[cal_commune['culture'] == culture]
    if filtre is not None:
        periodes = periodes[filtre(periodes)]
    
    nb = periodes[col_nb].iloc[0] if not periodes.empty else 0
    ift_par_periode = ift_val / nb if nb > 0 else 0
    print(f"  {label:26s} {str(culture):20s} {ift_val:>10.3f} {nb:>12} {ift_par_periode:>12.4f}")

## 4. Simulation du risque jour par jour sur l'année

In [ ]:
from calcul_risque_journalier import compute_ift_journalier, load_data

ift_data, cal_data, meteo = load_data(ANNEE)

# Simuler jour par jour pour la commune
nb_jours = 366 if ANNEE % 4 == 0 else 365
dates    = [date(ANNEE, 1, 1) + timedelta(days=d) for d in range(nb_jours)]

resultats = []
for d in dates:
    ift_j = compute_ift_journalier(ift_data, cal_data, d)
    row_j = ift_j.filter(pl.col('insee_com') == INSEE)
    ift_val = row_j['ift_journalier_total'][0] if len(row_j) > 0 else 0.0
    resultats.append({'date': d, 'ift_journalier': ift_val})

ts = pd.DataFrame(resultats)
print(f"IFT journalier max : {ts['ift_journalier'].max():.4f}")
print(f"Jours avec IFT > 0 : {(ts['ift_journalier'] > 0).sum()}")
print(f"Somme IFT annuel reconstituée : {ts['ift_journalier'].sum():.3f}")
print(f"IFT annuel source (ift_t_hbc) : {r.get('ift_t_hbc', 0):.3f}")

## 5. Visualisation

In [ ]:
# Palette risque
RISK_COLORS = {0: '#A5D6A7', 1: '#FFF176', 2: '#FFB300', 3: '#F57C00', 4: '#B71C1C'}

# Normalisation 0-4 simplifiée (quartiles)
vals_pos = ts[ts['ift_journalier'] > 0]['ift_journalier']
if len(vals_pos) > 0:
    q1, q2, q3 = vals_pos.quantile([0.25, 0.5, 0.75])
    def niveau(v):
        if v == 0: return 0
        if v <= q1: return 1
        if v <= q2: return 2
        if v <= q3: return 3
        return 4
    ts['risque_0_4'] = ts['ift_journalier'].apply(niveau)
else:
    ts['risque_0_4'] = 0
ts['color'] = ts['risque_0_4'].map(RISK_COLORS)

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[
        f'IFT journalier — {r.get("nom_commune", INSEE)} ({INSEE})',
        'Indicateur de risque 0-4'
    ],
    vertical_spacing=0.08
)

# IFT journalier
fig.add_trace(go.Bar(
    x=ts['date'], y=ts['ift_journalier'],
    marker_color=ts['color'],
    hovertemplate='<b>%{x|%d/%m/%Y}</b><br>IFT : %{y:.4f}<extra></extra>',
    name='IFT journalier'
), row=1, col=1)

# Périodes de traitement du calendrier (vigne dep 37)
colors_type = {'Herbicides': 'rgba(255,100,0,0.15)', 'Fongi/Insecti': 'rgba(0,100,255,0.1)'}
for _, p in cal_commune[cal_commune['culture'].isin(cultures_commune)].iterrows():
    label_p = 'Herbicides' if p['Herbicides'] else 'Fongi/Insecti'
    fig.add_vrect(
        x0=str(p['debut']), x1=str(p['fin']),
        fillcolor=colors_type.get(label_p, 'rgba(128,128,128,0.1)'),
        opacity=1, line_width=0,
        annotation_text=f"{p['culture'][:4]} {label_p[:4]}",
        annotation_font_size=9,
        row=1, col=1
    )

# Risque 0-4
fig.add_trace(go.Bar(
    x=ts['date'], y=ts['risque_0_4'],
    marker_color=ts['color'],
    hovertemplate='<b>%{x|%d/%m/%Y}</b><br>Risque : %{y}<extra></extra>',
    name='Risque 0-4', showlegend=False
), row=2, col=1)

fig.update_layout(
    height=600, bargap=0,
    hovermode='x unified',
    margin=dict(l=50, r=20, t=50, b=20),
    showlegend=False
)
fig.update_yaxes(title_text='IFT/jour', row=1)
fig.update_yaxes(title_text='Risque', tickvals=[0,1,2,3,4], range=[0,4.3], row=2)
fig.show()

## 6. Détail par période de traitement

In [ ]:
# Pour chaque période du calendrier, afficher l'IFT distribué et la cohérence
print(f"{'Culture':15s} {'Période':25s} {'Herb':5s} {'Fongi':5s} {'Insec':5s} {'IFT/j (total)':>14} {'IFT/j (h)':>10} {'IFT/j (f+i)':>12}")
print('-' * 100)

for _, p in cal_commune[cal_commune['culture'].isin(cultures_commune)].sort_values(['culture','debut']).iterrows():
    # IFT journalier prévu pour une date dans cette période
    test_date = p['debut'] + timedelta(days=1)
    ift_j = compute_ift_journalier(ift_data, cal_data, test_date)
    row_j  = ift_j.filter(pl.col('insee_com') == INSEE)
    ift_val = row_j['ift_journalier_total'][0] if len(row_j) > 0 else 0.0
    
    print(
        f"  {p['culture']:13s} "
        f"{str(p['debut'])} → {str(p['fin'])}  "
        f"{'✓' if p['Herbicides'] else ' ':5s}"
        f"{'✓' if p['Fongicides'] else ' ':5s}"
        f"{'✓' if p['Insecticides'] else ' ':5s}"
        f"  {ift_val:>12.4f}"
    )